In [0]:
%scala
println("Hello World")

Hello Worlds


In [0]:
%sql
DESCRIBE EXTERNAL LOCATION `adlsdatabricks2-exloc`;

name,url,credential_name,owner,created_by,created_at,comment
adlsdatabricks2-exloc,abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/,adls_storage_credential,jaswanthibevara@gmail.com,jaswanthibevara@gmail.com,2026-11-16T13:35:48.009Z,null


In [0]:
%sql
CREATE CATALOG IF NOT EXISTS orders_demo
MANAGED LOCATION
'abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/managed/orders_demo';
CREATE SCHEMA  IF NOT EXISTS orders_demo.bronze;
CREATE SCHEMA  IF NOT EXISTS orders_demo.silver;


In [0]:
%scala
val rawPath =
  "abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/"

dbutils.fs.ls(rawPath).foreach(println)

FileInfo(abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/Orders_2024_01_15.txt, Orders_2024_01_15.txt, 219, 1794837736000)


rawPath: String = "abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/"

In [0]:

%scala

import org.apache.spark.sql.functions._
import org.apache.spark.sql.types._

// paths 

val rawPath =
  "abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/Orders_2024_01_15.txt"

val bronzeTable = "orders_demo.bronze.orders_raw"

// read CSV

val dfRaw = spark.read
  .format("csv")
  .option("header", "true")
  .option("inferSchema", "true")
  .load(rawPath)

// light audit columns

val dfBronze = dfRaw
  .withColumn("ingest_ts", current_timestamp())
  .withColumn("source_file", col("_metadata.file_path"))

//write as Delta (append)

val rowCount = dfBronze.count()

dfBronze.write
  .format("delta")
  .mode("append")
  .option("mergeSchema", "true")
  .saveAsTable(bronzeTable)

println(s"Landed $rowCount rows into $bronzeTable")


Landed 5 rows into orders_demo.bronze.orders_raw


import org.apache.spark.sql.functions._
import org.apache.spark.sql.types._
rawPath: String = "abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/Orders_2024_01_15.txt"
bronzeTable: String = "orders_demo.bronze.orders_raw"
dfRaw: org.apache.spark.sql.package.DataFrame = [order_id: int, customer_id: int ... 3 more fields]
dfBronze: org.apache.spark.sql.package.DataFrame = [order_id: int, customer_id: int ... 5 more fields]
rowCount: Long = 5L

In [0]:
%sql
CREATE TABLE IF NOT EXISTS orders_demo.bronze.orders_raw
USING DELTA
LOCATION 'abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/bronze/orders_raw';

In [0]:
%scala

import org.apache.spark.sql.functions._
import org.apache.spark.sql.types._

val bronzeTable = "orders_demo.bronze.orders_raw"
val silverTable = "orders_demo.silver.orders"

// cleaning and transformations 

val dfClean = spark.table(bronzeTable)

  // Keep only valid order IDs
  .filter(col("order_id").isNotNull)

  // Remove zero or negative amounts
  .filter(col("amount") > 0)

  // Convert date string to DATE
  .withColumn("order_date", to_date(col("order_date")))

  // Convert amount to DECIMAL(12,2)
  .withColumn("amount", col("amount").cast(DecimalType(12, 2)))

  // Normalize status
  .withColumn("status", upper(trim(col("status"))))

  // Remove duplicate order IDs
  .dropDuplicates(Seq("order_id"))

// Register temporary view
dfClean.createOrReplaceTempView("orders_clean")

// Verify cleaned records
dfClean.show(false)

+-----+--------------------------+----------------------------------------------------------------------------------+--------+-----------+----------+------+---------+
|value|ingest_ts                 |source_file                                                                       |order_id|customer_id|order_date|amount|status   |
+-----+--------------------------+----------------------------------------------------------------------------------+--------+-----------+----------+------+---------+
|NULL |2026-11-16 15:59:10.528905|abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/Orders_2024_01_15.txt|101     |1          |2024-01-10|250.00|COMPLETED|
|NULL |2026-11-16 15:59:10.528905|abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/Orders_2024_01_15.txt|102     |1          |2024-01-12|180.50|COMPLETED|
|NULL |2026-11-16 15:59:10.528905|abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/Orders_2024_01_15.txt|103     |2          |2024-01-14|99.99 |CANCELLED

import org.apache.spark.sql.functions._
import org.apache.spark.sql.types._
bronzeTable: String = "orders_demo.bronze.orders_raw"
silverTable: String = "orders_demo.silver.orders"
dfClean: org.apache.spark.sql.Dataset[Row] = [value: string, ingest_ts: timestamp ... 6 more fields]

In [0]:
%scala

import org.apache.spark.sql.functions._

//  target table

val silverTable = "orders_demo.silver.orders"

//  create table on first run 

spark.sql(s"""
  CREATE TABLE IF NOT EXISTS $silverTable (
    order_id BIGINT,
    customer_id INT,
    order_date DATE,
    amount DECIMAL(12,2),
    status STRING,
    updated_at TIMESTAMP
  ) USING DELTA
""")

//  MERGE: upsert

spark.sql(s"""
  MERGE INTO $silverTable AS t
  USING orders_clean AS s
  ON t.order_id = s.order_id

  WHEN MATCHED THEN UPDATE SET
    t.customer_id = s.customer_id,
    t.order_date = s.order_date,
    t.amount = s.amount,
    t.status = s.status,
    t.updated_at = current_timestamp()

  WHEN NOT MATCHED THEN INSERT (
    order_id,
    customer_id,
    order_date,
    amount,
    status,
    updated_at
  )
  VALUES (
    s.order_id,
    s.customer_id,
    s.order_date,
    s.amount,
    s.status,
    current_timestamp()
  )
""")

println("Silver MERGE completed successfully!")

Silver MERGE completed successfully!


import org.apache.spark.sql.functions._
silverTable: String = "orders_demo.silver.orders"
res22_2: org.apache.spark.sql.package.DataFrame = []
res22_3: org.apache.spark.sql.package.DataFrame = [num_affected_rows: bigint, num_updated_rows: bigint ... 2 more fields]

In [0]:
%scala

import io.delta.tables._
import org.apache.spark.sql.functions._

val deltaTarget = DeltaTable.forName(spark, silverTable)

deltaTarget.as("t")
  .merge(
    dfClean.as("s"),
    "t.order_id = s.order_id"
  )
  .whenMatched()
  .updateExpr(Map(
    "customer_id" -> "s.customer_id",
    "order_date" -> "s.order_date",
    "amount" -> "s.amount",
    "status" -> "s.status",
    "updated_at" -> "current_timestamp()"
  ))
  .whenNotMatched()
  .insertExpr(Map(
    "order_id" -> "s.order_id",
    "customer_id" -> "s.customer_id",
    "order_date" -> "s.order_date",
    "amount" -> "s.amount",
    "status" -> "s.status",
    "updated_at" -> "current_timestamp()"
  ))
  .execute()

import io.delta.tables._
import org.apache.spark.sql.functions._
deltaTarget: DeltaTable = io.delta.tables.DeltaTable@4a1bfdf9
res23_3: org.apache.spark.sql.package.DataFrame = [num_affected_rows: bigint, num_updated_rows: bigint ... 2 more fields]

In [0]:
%sql

DESCRIBE HISTORY orders_demo.silver.orders;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
2,2026-11-16T16:03:16Z,143925882962039,jaswanthibevara@gmail.com,MERGE,"Map(predicate -> [""(order_id#1302L = cast(order_id#828 as bigint))""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> false, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}], clusteringOnWriteStatus -> null)",null,List(90119396896719),null,1116-134623-cb5nmgm3,1,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 1, numTargetBytesAdded -> 3258, numTargetBytesRemoved -> 2429, numTargetDeletionVectorsAdded -> 0, numTargetRowsMatchedUpdated -> 5, executionTimeMs -> 4154, materializeSourceTimeMs -> 735, numTargetRowsInserted -> 0, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 1797, numTargetRowsUpdated -> 5, numOutputRows -> 5, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 5, numTargetFilesRemoved -> 1, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 1567)",null,Databricks-Runtime/18.x-scala2.13
1,2026-11-16T16:02:47Z,143925882962039,jaswanthibevara@gmail.com,MERGE,"Map(predicate -> [""(order_id#861L = cast(order_id#550 as bigint))""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> false, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}], clusteringOnWriteStatus -> null)",null,List(90119396896719),null,1116-134623-cb5nmgm3,0,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 1, numTargetBytesAdded -> 2429, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 0, numTargetRowsMatchedUpdated -> 0, executionTimeMs -> 4455, materializeSourceTimeMs -> 1008, numTargetRowsInserted -> 5, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 2536, numTargetRowsUpdated -> 0, numOutputRows -> 5, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 5, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 819)",null,Databricks-Runtime/18.x-scala2.13
0,2026-11-16T16:02:41Z,143925882962039,jaswanthibevara@gmail.com,CREATE TABLE,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""databricks.internal.autoUpgrades.delta.enableRowTracking"":""true"",""delta.enableDeletionVectors"":""true"",""databricks.internal.autoUpgrades.delta.checkpointPolicy"":""v2"",""databricks.internal.autoUpgrades.delta.parquet.format.version"":""2.12.0"",""delta.parquet.format.version"":""2.12.0"",""delta.enableRowTracking"":""true"",""delta.checkpointPolicy"":""v2"",""io.unitycatalog.tableId"":""f103c2a7-3aed-497d-878f-5976e5319efc"",""delta.rowTracking.materializedRowCommitVersionColumnName"":""_row-commit-version-col-7f0aa909-26a5-406f-ad5c-012f196f2dda"",""delta.rowTracking.materializedRowIdColumnName"":""_row-id-col-f25599e8-4719-4cf6-a7b8-2204aae25f88""}, statsOnLoad -> false, clusterByAuto -> true)",null,List(90119396896719),null,1116-134623-cb5nmgm3,null,WriteSerializable,true,Map(),null,Databricks-Runtime/18.x-scala2.13


In [0]:
%scala
spark.sql("""
SELECT *
FROM orders_demo.silver.orders
VERSION AS OF 0
""").show(false)

+--------+-----------+----------+------+------+----------+
|order_id|customer_id|order_date|amount|status|updated_at|
+--------+-----------+----------+------+------+----------+
+--------+-----------+----------+------+------+----------+


In [0]:
%scala
spark.sql("""
SELECT *
FROM orders_demo.silver.orders
VERSION AS OF 2
""").show(false)

+--------+-----------+----------+------+---------+--------------------------+
|order_id|customer_id|order_date|amount|status   |updated_at                |
+--------+-----------+----------+------+---------+--------------------------+
|101     |1          |2024-01-10|250.00|COMPLETED|2026-11-16 16:03:11.601223|
|102     |1          |2024-01-12|180.50|COMPLETED|2026-11-16 16:03:11.601223|
|103     |2          |2024-01-14|99.99 |CANCELLED|2026-11-16 16:03:11.601223|
|104     |3          |2024-01-15|320.00|COMPLETED|2026-11-16 16:03:11.601223|
|105     |3          |2024-01-15|150.00|PENDING  |2026-11-16 16:03:11.601223|
+--------+-----------+----------+------+---------+--------------------------+


In [0]:
%scala
spark.sql("""
SELECT *
FROM orders_demo.silver.orders
TIMESTAMP AS OF '2026-11-16 16:02:47'
""").show(false)

+--------+-----------+----------+------+---------+--------------------------+
|order_id|customer_id|order_date|amount|status   |updated_at                |
+--------+-----------+----------+------+---------+--------------------------+
|101     |1          |2024-01-10|250.00|COMPLETED|2026-11-16 16:02:42.653498|
|102     |1          |2024-01-12|180.50|COMPLETED|2026-11-16 16:02:42.653498|
|103     |2          |2024-01-14|99.99 |CANCELLED|2026-11-16 16:02:42.653498|
|104     |3          |2024-01-15|320.00|COMPLETED|2026-11-16 16:02:42.653498|
|105     |3          |2024-01-15|150.00|PENDING  |2026-11-16 16:02:42.653498|
+--------+-----------+----------+------+---------+--------------------------+


In [0]:
%scala
val oldDF = spark.read
  .format("delta")
  .option("versionAsOf", 1)
  .table("orders_demo.silver.orders")

oldDF.show(false)

+--------+-----------+----------+------+---------+--------------------------+
|order_id|customer_id|order_date|amount|status   |updated_at                |
+--------+-----------+----------+------+---------+--------------------------+
|101     |1          |2024-01-10|250.00|COMPLETED|2026-11-16 16:02:42.653498|
|102     |1          |2024-01-12|180.50|COMPLETED|2026-11-16 16:02:42.653498|
|103     |2          |2024-01-14|99.99 |CANCELLED|2026-11-16 16:02:42.653498|
|104     |3          |2024-01-15|320.00|COMPLETED|2026-11-16 16:02:42.653498|
|105     |3          |2024-01-15|150.00|PENDING  |2026-11-16 16:02:42.653498|
+--------+-----------+----------+------+---------+--------------------------+


oldDF: org.apache.spark.sql.package.DataFrame = [order_id: bigint, customer_id: int ... 4 more fields]

In [0]:
%sql
RESTORE TABLE orders_demo.silver.orders
TO VERSION AS OF 0;RESTORE TABLE orders_demo.silver.orders
TO VERSION AS OF 0;

In [0]:
%scala
val bronzeTable = "orders_demo.bronze.orders_raw"

spark.sql("""
ALTER TABLE orders_demo.silver.orders
ADD COLUMN discount DECIMAL(5,2)
""")

bronzeTable: String = "orders_demo.bronze.orders_raw"
res28_1: org.apache.spark.sql.package.DataFrame = []

In [0]:
%scala
spark.table("orders_demo.silver.orders").printSchema()

root
 |-- order_id: long (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- order_date: date (nullable = true)
 |-- amount: decimal(12,2) (nullable = true)
 |-- status: string (nullable = true)
 |-- updated_at: timestamp (nullable = true)
 |-- discount: decimal(5,2) (nullable = true)



In [0]:
spark.table("orders_demo.bronze.orders_raw").printSchema()

root
 |-- value: string (nullable = true)
 |-- ingest_ts: timestamp (nullable = true)
 |-- source_file: string (nullable = true)
 |-- order_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- order_date: date (nullable = true)
 |-- amount: double (nullable = true)
 |-- status: string (nullable = true)



In [0]:
%scala
spark.sql("""
OPTIMIZE orders_demo.silver.orders
""").show(false)

+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|path                                                                                                                                                                                     |metrics                                                                                                                                                                        |
+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-------------------------------------------------------------------------------------

In [0]:
%scala
spark.sql("""
OPTIMIZE orders_demo.silver.orders
ZORDER BY (customer_id, order_date)
""").show(false)

+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|path                                                                                                                                                                                     |metrics                                                                                                                                                                                                                                       |
+-------------------------------------------------------------------------------------------------------------------------------------------------

In [0]:
%scala
spark.sql("""
VACUUM orders_demo.silver.orders DRY RUN
""").show(false)

+----+
|path|
+----+
+----+


In [0]:
%scala
spark.sql("""
VACUUM orders_demo.silver.orders RETAIN 168 HOURS
""").show(false)

+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|path                                                                                                                                                                                     |
+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|abfss://adlsdatabrics2c@adlsdatabricks2.dfs.core.windows.net/managed/orders_demo/__unitystorage/catalogs/a2dec286-7a60-4d34-b068-ea9e8fc47e6a/tables/f103c2a7-3aed-497d-878f-5976e5319efc|
+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
